# 04 · 스트리밍과 완료 계약

SSE 이벤트 형식과 실패·취소를 다룹니다. 권장 15분. ASGITransport는 응답을 버퍼링하므로 첫 조각 지연을 측정하지 않습니다.

위에서 아래로 실행합니다. 각 실험은 새 앱을 사용하며 실제 LLM·GPU·외부 API는 기본 경로에 없습니다. 먼저 결과를 예상하고 실행한 뒤 변수 하나를 바꾸고 관찰을 기록하세요.

In [ ]:
from pathlib import Path
import sys, asyncio, json, time
from dataclasses import replace
import numpy as np
import httpx
from IPython.display import display
ROOT = next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'course_labs').is_dir())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from week09.lab.app import create_app
from week09.lab.service import Settings, ChatRequest, Service, TTLCache, TokenBucket, cache_key
from week09.lab.benchmark import run_load, summarize
assert sys.version_info[:2] == (3,11), 'Python 3.11 .venv 커널 선택'
OUT = ROOT/'output/validation/week09-lab'
OUT.mkdir(parents=True,exist_ok=True)
print('Python',sys.version.split()[0],'NumPy',np.__version__,'HTTPX',httpx.__version__)


## SSE 프레임

event와 data 뒤의 빈 줄이 한 프레임의 끝입니다. 실제 LLM 토큰이 아닌 문자 조각입니다.

In [ ]:
frame = 'event: delta\ndata: {"text":"hello"}\n\n'
assert frame.endswith('\n\n')
print(frame)

## 프레임 파서

가독성을 위한 완성 응답 파서입니다. 브라우저 예제는 네트워크 조각 경계를 따로 누적합니다.

In [ ]:
def parse_events(text):
    rows = []
    for block in text.strip().split('\n\n'):
        fields = dict(line.split(':',1) for line in block.splitlines() if ':' in line)
        if 'data' in fields:
            rows.append({'event':fields.get('event','message').strip(),'data':json.loads(fields['data'])})
    return rows
assert parse_events(frame)[0]['data']['text'] == 'hello'

## 정상 스트림

meta 이후 delta들을 합치고 done을 확인합니다. HTTP 200만으로 완료 처리하지 않습니다.

In [ ]:
app = create_app(Settings(mock_delay_s=.001,timeout_s=.1))
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    r = await client.post('/api/v1/chat/stream',json={'message':'반품 정책'})
events = parse_events(r.text)
assert events[0]['event'] == 'meta' and events[-1]['event'] == 'done'
answer = ''.join(x['data']['text'] for x in events if x['event']=='delta')
assert '[합성 정책 예시]' in answer
display(events)

## 헤더 이후의 실패

응답이 이미 시작됐으므로 오류 이벤트로 미완료를 알립니다. 상태 코드는 200으로 남습니다.

In [ ]:
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    r = await client.post('/api/v1/chat/stream',json={'message':'x','scenario':'timeout'})
failed_events = parse_events(r.text)
assert r.status_code == 200 and failed_events[-1]['event'] == 'error'
assert not any(e['event']=='done' for e in failed_events)
assert app.state.service.active == 0
display(failed_events)

## 상류 오류와 사용자 표시

불완전 답변을 성공 결과로 저장하지 않습니다.

In [ ]:
async with httpx.AsyncClient(transport=httpx.ASGITransport(app=app),base_url='http://lab') as client:
    r = await client.post('/api/v1/chat/stream',json={'message':'x','scenario':'upstream_error'})
assert parse_events(r.text)[-1]['data']['complete'] is False
print('화면: 실패 사유·재시도 가능 여부·다음 행동')

## 중도 종료와 자원

생성기를 닫았을 때도 finally에서 active가 해제돼야 합니다.

In [ ]:
class Connected:
    async def is_disconnected(self): return False
service = Service(Settings(mock_delay_s=.001))
service.admit()
generator = service.stream(ChatRequest(message='x'),'manual-close',Connected())
print(await anext(generator))
await generator.aclose()
assert service.active == 0

## 첫 응답과 첫 내용

meta 도착을 첫 토큰 도착으로 측정하면 빠르게 보이는 착시가 생깁니다.

In [ ]:
timing_spec = {'first_byte':'첫 바이트','first_delta':'첫 내용 조각','complete':'done 수신','transport':'실제 로컬 TCP에서 별도 측정'}
display(timing_spec)

## 실제 서버로 확장

터미널에서 서버를 띄운 후 curl -N 또는 브라우저로 확인합니다. 자동 실행은 외부 서버를 호출하지 않습니다.

In [ ]:
print('python -m uvicorn week09.lab.app:app --host 127.0.0.1 --port 8009')
print('브라우저: http://127.0.0.1:8009/ → 스트리밍 → 수신 중단')
print('별도 검증: python scripts/verify_week09_live.py')

## 완료 계약 기록

SSE는 웹소켓과 다른 단방향 전달 형식입니다. POST는 fetch를 사용하며 기본 EventSource에 POST body를 넣지 않습니다.

In [ ]:
contract = {'normal':['meta','delta','done'],'failed':['meta','error'],'cancelled':'done 없음, 자원 해제','transport_test':'실제 TCP 필요'}
(OUT/'stream-contract.json').write_text(json.dumps(contract,ensure_ascii=False,indent=2),encoding='utf-8')
display(contract)

## 포트폴리오 기록

변경 변수 / 고정 조건 / 실제 관측값 / 실패 사례 / 해석의 한계 / 다음 확인을 기록하세요. 개인정보·학생 원문은 공개 저장소에 넣지 않습니다. 별도 과제 마감은 소속 공지를 따르며 이 실습에서 새 제출 기한을 만들지 않습니다.